### Inserting the raw data into the bronze layer


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

data = [
        (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1), 
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)

]
columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]

df_raw1= spark.createDataFrame(data, columns)
df_raw1.write.format("delta").mode("append").save("/Volumes/medallion_architecture/bronze/rawdata")

### reading from the bronze layer

In [0]:
df_bronze1 = spark.read.format("delta").load("/Volumes/medallion_architecture/bronze/rawdata")
df_bronze1.display()


In [0]:
df_silver1 = df_bronze1.withColumn("amount",col("amount").cast("long")).withColumn("date",col("date").cast("date"))
df_silver1.display()


In [0]:
df_silver1= df_silver1.fillna({"amount":18000,"city":"unknown"})
df_silver1.display()

In [0]:
df_silver1=df_silver1.filter(col("amount")>0)
df_silver1.display()

In [0]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy("order_id").orderBy(col("date").desc())

df_silver1 = df_silver1.withColumn("rank", row_number().over(window_spec)) \
                     .filter(col("rank") == 1) \
                     .drop("rank")
        

In [0]:
df_silver1.display()

In [0]:
df_silver1.dropDuplicates().display()


In [0]:
df_silver1.write.format("delta").mode("overwrite").save("/Volumes/medallion_architecture/silver/cleandata")

###Gold layer

In [0]:
df_gold1 = spark.read.format("delta").load("/Volumes/medallion_architecture/silver/cleandata")
df_gold1.display()

In [0]:
df_gold1.write.format("delta").mode("overwrite").save("/Volumes/medallion_architecture/gold/analyseddata")

In [0]:
df_gold1.printSchema()

In [0]:
df_gold1.groupBy("product").agg(sum("amount")).display()

In [0]:
df_gold1.groupBy("category").agg(sum("amount")).display()

In [0]:
df_gold1.groupBy("city").agg(sum("amount")).display()

In [0]:
df_gold1.groupBy("customer_id").agg(count("order_id")).display()

In [0]:
df_gold1.groupBy("customer_id").agg(sum("amount")).display()

In [0]:
from pyspark.sql.functions import sum, col

df_top_product = df_gold1.groupBy("product") \
    .agg(sum("amount").alias("total_sales")) \
    .orderBy(col("total_sales").desc()) \
    .limit(1).display()

In [0]:
from pyspark.sql.functions import sum, col

df_top_customer = df_gold1.groupBy("customer_id") \
    .agg(sum("amount").alias("total_spent")) \
    .orderBy(col("total_spent").desc()) \
    .limit(1).display()